<a href="https://colab.research.google.com/github/hhy37/-ExcelVBA/blob/master/MPI_on_Terminal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Initialization and Environment Setup
Run this cell to install `mpi4py` and prepare the workspace for MPI communication.

In [19]:
import sys
import os

# Install mpi4py if not present
!{sys.executable} -m pip install -q mpi4py

# Generate the demo script automatically
with open('mpi_demo.py', 'w') as f:
    f.write("""\
from mpi4py import MPI
import numpy as np
import sys

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

if size < 2:
    if rank == 0: print('Need at least 2 processes.')
    sys.exit(0)

if rank == 0:
    data = np.array([42], dtype=int)
    print(f'Rank 0: Sending {data[0]} to Rank 1')
    comm.Send(data, dest=1, tag=11)
elif rank == 1:
    data = np.empty(1, dtype=int)
    comm.Recv(data, source=0, tag=11)
    print(f'Rank 1: Received {data[0]} from Rank 0')
""")

print("Environment initialized. You can now run the mpiexec commands below.")

Environment initialized. You can now run the mpiexec commands below.


In [13]:
import sys
!{sys.executable} -m pip install mpi4py
from mpi4py import MPI
import numpy as np

# This gives you MPI_COMM_WORLD
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size() # Get the number of processes

# Initialize variables
numbertosend = np.array([42], dtype=int)
numbertoreceive = np.empty(1, dtype=int)

# Check if there are enough processes for Send/Recv communication
if size < 2:
    print(f"MPI communication requires at least 2 processes to demonstrate Send/Recv between ranks. Current number of processes: {size}. Skipping communication part.", file=sys.stderr)
else:
    # ---------------------------------------------------------
    # Equivalent to:
    # call MPI_Send(numbertosend, 1, MPI_INTEGER, index, 10, MPI_COMM_WORLD, errcode)
    # ---------------------------------------------------------
    if rank == 0:
        index = 1 # Destination rank
        comm.Send(numbertosend, dest=index, tag=10)
        print(f"Rank {rank}: Sent {numbertosend[0]} to rank {index}.")

    # ---------------------------------------------------------
    # Equivalent to:
    # call MPI_Recv(numbertoreceive, 1, MPI_INTEGER, 0, 10, MPI_COMM_WORLD, status, errcode)
    # ---------------------------------------------------------
    elif rank == 1:
        comm.Recv(numbertoreceive, source=0, tag=10)
        print(f"Rank {rank}: Received {numbertoreceive[0]} from rank 0.")

# ---------------------------------------------------------
# Equivalent to:
# call MPI_Barrier(MPI_COMM_WORLD, errcode)
# ---------------------------------------------------------
comm.Barrier()

MPI communication requires at least 2 processes to demonstrate Send/Recv between ranks. Current number of processes: 1. Skipping communication part.


### Running MPI with Multiple Processes
To actually see `Send` and `Recv` in action, we must use the `mpiexec` command to launch multiple processes.
1. We use `%%writefile` to save the Python code to a file.
2. We use `!mpiexec` to run that file with multiple ranks.

In [15]:
%%writefile mpi_demo.py
from mpi4py import MPI
import numpy as np
import sys

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

if size < 2:
    print(f"Process {rank}: Need at least 2 processes.")
    sys.exit(0)

if rank == 0:
    data = np.array([42], dtype=int)
    print(f"Rank 0: Sending {data[0]} to Rank 1")
    comm.Send(data, dest=1, tag=11)
elif rank == 1:
    data = np.empty(1, dtype=int)
    comm.Recv(data, source=0, tag=11)
    print(f"Rank 1: Received {data[0]} from Rank 0")


Overwriting mpi_demo.py


In [16]:
# Run the script using 2 processes
!mpiexec --allow-run-as-root -n 2 python mpi_demo.py

### Checking for NPU Hardware
This script attempts to identify NPU hardware by checking system logs and PCI devices. Note that in a Google Colab environment, this will reflect Google's server hardware, not your local laptop. To check your laptop, you should run these steps in your local terminal or Task Manager.

In [11]:
import subprocess

def check_npu():
    print("--- Checking for NPU hardware signatures ---")
    # Check for common NPU strings in PCI devices (Linux/Colab environment)
    try:
        lscpu = subprocess.check_output("lscpu", shell=True).decode()
        print(f"CPU Info: {lscpu.split('\n')[0]}")

        # Search for 'NPU', 'VPU', or 'Accelerator' in system logs
        dmesg = subprocess.check_output("dmesg | grep -iE 'npu|vpu|accel'", shell=True).decode()
        if dmesg:
            print("Potential NPU/Accelerator found in logs:")
            print(dmesg)
        else:
            print("No explicit NPU device detected in this environment.")
    except Exception as e:
        print(f"Could not run hardware check: {e}")

check_npu()

--- Checking for NPU hardware signatures ---
CPU Info: Architecture:                            x86_64
Potential NPU/Accelerator found in logs:
[    0.939990] input: Power Button as /devices/LNXSYSTM:00/LNXPWRBN:00/input/input0
[    0.942353] input: Sleep Button as /devices/LNXSYSTM:00/LNXSLPBN:00/input/input1
[    1.294721] input: AT Translated Set 2 keyboard as /devices/platform/i8042/serio0/input/input2
[   14.602895]     input device check on



### Checking the Supported MPI Standard Version
This code identifies which version of the MPI specification (e.g., 3.1, 4.0) your current `mpi4py` installation is using.

In [17]:
from mpi4py import MPI

# Get the MPI standard version
major, minor = MPI.Get_version()

print(f"Current MPI Standard Version: {major}.{minor}")
print(f"Library Vendor: {MPI.get_vendor()}")

Current MPI Standard Version: 3.1
Library Vendor: ('Open MPI', (4, 1, 2))


### Checking for Available Updates
This command checks the Python Package Index for the latest available version of `mpi4py`. While it might not be MPI-5 yet, staying updated ensures you have the latest features of the MPI standard.

In [18]:
!pip list --outdated | grep mpi4py || echo "mpi4py is already up to date."

mpi4py is already up to date.
